# MCP 第 07 课：Shell / CLI 工具，为什么它既强大又危险

如果说 API 工具是专用工具，那么 shell/CLI 几乎就是“通用超工具”。

它强的地方在于：

- 能读文件
- 能搜代码
- 能跑测试
- 能调用现成命令行程序

但它危险也正危险在这里。


## 1. Shell 工具的基本模式

在 agent 系统里，shell 工具通常不是“模型自己执行命令”，而是：

1. 模型提出命令
2. 宿主程序决定是否执行
3. 把输出再返回给模型

OpenAI 官方也把 `shell` 设计成这种模式，而不是让模型直接接管机器。


In [ ]:
allowed_examples = [
    "pwd",
    "rg --files src",
    "pytest tests/test_api.py",
    "git status --short",
]

allowed_examples


## 2. 为什么 CLI 对 coding agent 特别重要

因为代码工作天然依赖大量命令行能力：

- 搜索：`rg`
- 构建：`npm build` / `cargo build`
- 测试：`pytest` / `npm test`
- 版本控制：`git status` / `git diff`
- 格式化：`ruff format` / `prettier`

没有 CLI，coding agent 的“手脚”会少很多。


## 3. 一个最小命令白名单心智模型

你可以把 shell tool 想成三层防线：

- 允许哪些命令
- 允许在哪些目录执行
- 是否需要人类审批

这三层缺任何一层，风险都会急剧上升。


In [ ]:
policy = {
    "allow": ["pwd", "ls", "rg", "pytest", "git status", "git diff"],
    "deny": ["rm -rf /", "curl | sh", "git push --force"],
    "require_confirmation": ["git commit", "git push", "pip install"],
}

policy


## 4. 读型 CLI 和写型 CLI

这是一个特别实用的分类：

- 读型：`ls`、`rg`、`cat`、`git status`
- 写型：`mv`、`git commit`、`sed -i`
- 高危型：`rm`、`sudo`、联网下载执行

实际系统里，这三类通常要配不同权限。


## 5. Codex / Claude Code 为什么都很重视 CLI

因为它们不是“回答编程问题”，而是“在真实开发环境里完成工作”。

一旦目标是完成真实工作，就离不开：

- 文件系统
- 终端
- 测试工具
- 包管理器
- git


## 6. CLI 不是万能的

shell 很强，但不代表所有能力都该塞给 shell。

当某个能力有明确结构时，单独做成专用工具往往更好：

- 参数更清晰
- 更容易校验
- 更容易审计
- 更容易做权限控制


## 本课工程直觉

1. **CLI 是 coding agent 的核心执行界面之一。**
2. **shell 工具最需要白名单、目录限制和审批流。**
3. **能专门化的能力，尽量别全部退化成“跑一条命令”。**
